In [ ]:
import os
import numpy as np
import json
from openai import OpenAI
from pathlib import Path
from sentence_transformers import SentenceTransformer
from typing import Tuple

# Initialize the OpenAI client with the base URL and API key
client = OpenAI(
    base_url="https://api.studio.nebius.com/v1/",
    api_key=os.getenv(
        "OPENAI_API_KEY"
    ),  # Retrieve the API key from environment variables
)
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")


def extract_json_from(json_file_path: Path) -> list[str]:
    # 폴더 경로 내 모든 json 파일을 읽어 데이터를 추출
    with open(json_file_path, "r") as f:
        nl_task_dict = json.load(f)
    return nl_task_dict


def embed_data(data: list[str], model: SentenceTransformer) -> list[list[float]]:
    # 데이터 임베딩
    return model.encode(data)


def compute_cosine_similarity(
    vec1: list[list[float]], vec2: list[list[float]]
) -> float:
    # 코사인 유사도 계산
    return np.dot(vec1, vec2) / (np.linalg.norm(vec1) * np.linalg.norm(vec2))


def semantic_search(
    query_vec: list[list[float]],
    references: list[list[float]],
    k: int = 5,
) -> list[Tuple[int, float]]:
    # 쿼리와 레퍼런스 데이터의 임베딩을 계산하고 코사인 유사도를 계산

    sim_scores = [
        (idx, compute_cosine_similarity(query_vec, ref_vec))
        for idx, ref_vec in enumerate(references)
    ]
    sorted_sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)[:k]
    top_k_idx = [idx for idx, score in sorted_sim_scores]
    return top_k_idx


refer_dict = extract_json_from(Path("assets/nl_task_db.json"))
input_nl = "cook a egg with critical dependency"
refer_nls = list(refer_dict.keys())

input_nl_vec = embed_data([input_nl], model)
refer_vec = embed_data(refer_nls, model)

top_k_idx = semantic_search(input_nl_vec, refer_vec, k=2)
top_k_dict = {refer_nls[idx]: refer_dict[refer_nls[idx]] for idx in top_k_idx}

for idx, task_info in enumerate(top_k_dict.items(), 1):
    task_nl, task_file_name = task_info
    few_shot_output = extract_json_from(Path(f"assets/tasks/{task_file_name}"))
    few_shot_prompt = f"""
    ### **Example {idx}**
    
    **Input**
    ```
    {task_nl}
    ```
    
    **Output**
    ```json
    {few_shot_output}
    ```
    
    ---
    """
    print(few_shot_prompt)

jcci_zeroshot/2025-03-20_08_29_48_make coffee and.json - 31.5= - 1.0=
jcci_zeroshot/2025-03-20_08_24_27_prepare vegetab.json - 29.2= - 0.5=
jcci_zeroshot/2025-03-20_08_34_16_make coffee and.json - 24.0= - 1.0=
jcci_zeroshot/2025-03-20_08_25_45_prepare vegetab.json - 17.2= - 1.0=
jcci_zeroshot/2025-03-20_08_33_01_make coffee and.json - 25.6= - 1.0=
jcci_zeroshot/2025-03-20_08_36_42_cook egg fry an.json - 30.2= - 1.0=
jcci_zeroshot/2025-03-20_08_39_19_cook egg fry an.json - 41.3= - 1.0=
jcci_zeroshot/2025-03-20_08_38_05_cook egg fry an.json - 25.6= - 1.0=
jcci_zeroshot/2025-03-20_08_35_33_make coffee and.json - 25.1= - 1.0=
jcci_zeroshot/2025-03-20_08_22_47_prepare vegetab.json - 46.5= - 1.0=
jcci_zeroshot/2025-03-20_08_41_09_cook egg fry an.json - 30.8= - 0.6666666666666666=
jcci_zeroshot/2025-03-20_08_26_44_prepare vegetab.json - 2.6= - 1.0=
jcci_zeroshot/2025-03-20_08_31_23_make coffee and.json - 35.6= - 0.5=

jcci_zeroshot= - 365.20000000000005= - 11.666666666666666= - 13=
jcci_top1/